# 第一章 LangChain与LangGraph框架认知

### 1.4.1 LangChain案例：简单文本生成

In [ ]:
# 1. 导入模块
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 2. 加载 .env 环境变量
load_dotenv()

# 3. 配置 API Key
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")

if not API_KEY:
    raise ValueError("未检测到 API_KEY，请检查 .env 文件是否配置正确")

# 4. 初始化大模型
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="deepseek-chat",
    temperature=0.3
)

# 5. 构造 Prompt（教学阶段用字符串更直观）
prompt = "请写一段50字左右的 AI 学习建议，语言简洁、实用，适合初学者。"

# 6. 调用模型
response = llm.invoke(prompt)

# 7. 输出结果
print("生成的学习建议：")
print(response.content)


### 1.4.2 LangGraph案例：基础工作流执行

In [ ]:
# 1. 导入需要的模块
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

# 4. 初始化大模型
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="deepseek-chat",
    temperature=0.3
)


In [ ]:

# 5. 定义 State
class WorkflowState(TypedDict, total=False): # 如何设定key？
    user_role: str
    original_advice: str
    simplified_advice: str

# 6. 定义节点
def generate_advice(state: WorkflowState):  # 该函数固定了主要prompt， 仅用WorkflowState['user_role']为参数
    prompt = f"给{state['user_role']}写一段50字左右的 AI 学习建议。"
    result = llm.invoke(prompt)
    return {"original_advice": result.content}

def simplify_advice(state: WorkflowState):
    prompt = f"把下面的学习建议精简到30字以内：{state['original_advice']}"
    result = llm.invoke(prompt)
    return {"simplified_advice": result.content}


In [ ]:
# 7. 构建工作流
workflow = StateGraph(WorkflowState) # 所以类别WorkflowState的定义是为了限定输入的key对应的内容。StateGraph后，在编译了之后，工作流会根据WorkflowState的定义来验证输入的key。

workflow.add_node("generate", generate_advice) # 方法的参数格式：'name', 'function'
workflow.add_node("simplify", simplify_advice)

workflow.add_edge(START, "generate")  # 定义graph的起始节点
workflow.add_edge("generate", "simplify")  # 定义graph的节点之间的连接
workflow.add_edge("simplify", END)  # 定义graph的结束节点


In [ ]:

app = workflow.compile() # 编译工作流 app 为最终执行的工作流

# 8. 执行
result = app.invoke({"user_role": "高校学生"}) # 传入的key必须与WorkflowState中定义的key一致

# 9. 输出
print("原始学习建议：")
print(result["original_advice"])
print("\n精简后学习建议：")
print(result["simplified_advice"])

# 注意
# 传入的参数必须与WorkflowState中定义的key一致，否则会报错。
# node的函数中必须使用WorkflowState中定义的key来访问输入参数。